In [7]:
from pathlib import Path
import csv
import math
from collections import defaultdict
import numpy as np

In [10]:
bbox_csv_path = Path('../data/raw/RAW_VINBIG/train.csv')

with bbox_csv_path.open(newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    abnormality_counts = Counter(row['class_name'] for row in reader)

print(f'Total annotated abnormalities: {sum(abnormality_counts.values()):,}')
print('\nAbnormalities per classification:')
for finding_label, count in abnormality_counts.most_common():
    print(f'{finding_label}: {count:,}')


Total annotated abnormalities: 67,914

Abnormalities per classification:
No finding: 31,818
Aortic enlargement: 7,162
Cardiomegaly: 5,427
Pleural thickening: 4,842
Pulmonary fibrosis: 4,655
Nodule/Mass: 2,580
Lung Opacity: 2,483
Pleural effusion: 2,476
Other lesion: 2,203
Infiltration: 1,247
ILD: 1,000
Calcification: 960
Consolidation: 556
Atelectasis: 279
Pneumothorax: 226


In [2]:
classes = {
    'Cardiomegaly': 1,
    'Pleural thickening': 2,
    'Pulmonary fibrosis': 3,
    'Pleural effusion': 4,
    'Nodule/Mass': 5,
    'Infiltration': 6,
    'Consolidation': 7,
    'Atelectasis': 8,
    'Pneumothorax': 9,
}

vinbig_train_csv_path = Path("../data/raw/RAW_VINBIG/train.csv")
vinbig_out_csv_path = Path("../data/processed/vinbig_train_selected_classes.csv")
vinbig_out_csv_path.parent.mkdir(parents=True, exist_ok=True)

total_rows = 0
kept_rows = 0
kept_per_class = {name: 0 for name in classes}

with vinbig_train_csv_path.open(newline="", encoding="utf-8") as infile, vinbig_out_csv_path.open(
    "w", newline="", encoding="utf-8"
) as outfile:
    reader = csv.DictReader(infile)

    out_fieldnames = list(reader.fieldnames or [])
    if "target_class_id" not in out_fieldnames:
        out_fieldnames.append("target_class_id")

    writer = csv.DictWriter(outfile, fieldnames=out_fieldnames)
    writer.writeheader()

    for row in reader:
        total_rows += 1
        class_name = (row.get("class_name") or "").strip()
        if class_name not in classes:
            continue

        row["target_class_id"] = classes[class_name]
        writer.writerow(row)

        kept_rows += 1
        kept_per_class[class_name] += 1

print(f"Read rows: {total_rows:,}")
print(f"Wrote rows (selected classes): {kept_rows:,}")
print(f"Output: {vinbig_out_csv_path.resolve()}")
print("\nRows per selected class:")
for name, _ in sorted(classes.items(), key=lambda kv: kv[1]):
    print(f"{name}: {kept_per_class[name]:,}")

Read rows: 67,914
Wrote rows (selected classes): 22,288
Output: D:\Home\Documents\GitHub\cxraide-data-pipeline\notebooks\data\processed\vinbig_train_selected_classes.csv

Rows per selected class:
Cardiomegaly: 5,427
Pleural thickening: 4,842
Pulmonary fibrosis: 4,655
Pleural effusion: 2,476
Nodule/Mass: 2,580
Infiltration: 1,247
Consolidation: 556
Atelectasis: 279
Pneumothorax: 226


In [4]:
# Input (your already-filtered VinBig CSV)
vinbig_csv_path = Path("../data/processed/vinbig_train_selected_classes.csv")

# Collect per-image info
bboxes_per_image = Counter()
diseases_per_image = defaultdict(set)
disease_bbox_counts_per_image = defaultdict(Counter)

with vinbig_csv_path.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        image_id = (row.get("image_id") or "").strip()
        class_name = (row.get("class_name") or "").strip()

        # A bbox exists if coords are present (No finding rows won't be in this file anyway)
        x_min = (row.get("x_min") or "").strip()
        y_min = (row.get("y_min") or "").strip()
        x_max = (row.get("x_max") or "").strip()
        y_max = (row.get("y_max") or "").strip()
        has_bbox = all([x_min, y_min, x_max, y_max])

        if not image_id:
            continue

        if has_bbox:
            bboxes_per_image[image_id] += 1

        if class_name:
            diseases_per_image[image_id].add(class_name)
            if has_bbox:
                disease_bbox_counts_per_image[image_id][class_name] += 1

print(f"Unique images: {len(diseases_per_image):,}")
print(f"Total bboxes (rows with coords): {sum(bboxes_per_image.values()):,}")

# Show images with the most bboxes
top_k = 20
print(f"\nTop {top_k} images by bbox count:")
for image_id, n in bboxes_per_image.most_common(top_k):
    diseases = ", ".join(sorted(diseases_per_image[image_id]))
    per_disease = "; ".join(
        f"{d}:{c}" for d, c in disease_bbox_counts_per_image[image_id].most_common()
    )
    print(f"{image_id} | bboxes={n} | diseases=[{diseases}] | per_disease=[{per_disease}]")

# Optional: inspect one specific image_id
# target_image_id = "PUT_IMAGE_ID_HERE"
# print("bboxes:", bboxes_per_image[target_image_id])
# print("diseases:", sorted(diseases_per_image[target_image_id]))
# print("per_disease:", disease_bbox_counts_per_image[target_image_id])

Unique images: 4,158
Total bboxes (rows with coords): 22,288

Top 20 images by bbox count:
03e6ecfa6f6fb33dfeac6ca4f9b459c9 | bboxes=55 | diseases=[Nodule/Mass] | per_disease=[Nodule/Mass:55]
fa109c087e46fe1ea27e48ce6d154d2f | bboxes=47 | diseases=[Consolidation, Nodule/Mass] | per_disease=[Nodule/Mass:46; Consolidation:1]
ecf474d5d4f65d7a3e23370a68b8c6a0 | bboxes=43 | diseases=[Cardiomegaly, Nodule/Mass, Pleural thickening, Pulmonary fibrosis] | per_disease=[Nodule/Mass:36; Pleural thickening:4; Cardiomegaly:2; Pulmonary fibrosis:1]
d106ec9b305178f3da060efe3191499a | bboxes=35 | diseases=[Nodule/Mass, Pleural effusion, Pleural thickening] | per_disease=[Nodule/Mass:33; Pleural thickening:1; Pleural effusion:1]
c699f16ba0b86f474390da9515bcad7a | bboxes=34 | diseases=[Cardiomegaly, Infiltration, Nodule/Mass, Pleural thickening, Pulmonary fibrosis] | per_disease=[Nodule/Mass:21; Pulmonary fibrosis:5; Cardiomegaly:3; Infiltration:3; Pleural thickening:2]
3a302fbbbf3364aa1a7731b59e6b98ec |

In [6]:
# ============================================================
# PURPOSE OF THIS CELL
# ============================================================
# This cell reads your original annotation CSV file, transforms
# the bounding box coordinates so they match images resized to
# 1024 x 1024, then saves the result as a new CSV file.
#
# It assumes your images were resized directly to 1024 x 1024.
# Meaning:
# - original image width  -> 1024
# - original image height -> 1024
#
# If your images were resized using aspect ratio + padding,
# this script should NOT be used because the bbox formula differs.
# ============================================================


# ============================================================
# CONFIGURATION
# ============================================================
# Change this to the filename/path of your original CSV file.
INPUT_CSV = "../data/processed/01_vinbig_train_selected_classes.csv"

# This will be the new CSV file containing transformed boxes.
OUTPUT_CSV = "../data/processed/02_vinbig_transformed_bounding_boxes_1024.csv"

# Target image size after resizing.
TARGET_SIZE = 1024


# ============================================================
# LOAD THE ORIGINAL CSV FILE
# ============================================================
# The CSV should contain these columns:
# x_min, y_min, x_max, y_max, width, height
#
# width  = original image width before resizing
# height = original image height before resizing
df = pd.read_csv(INPUT_CSV)


# ============================================================
# CALCULATE SCALE FACTORS
# ============================================================
# scale_x tells us how much the x-coordinates changed.
# Example:
# If original width is 2080 and new width is 1024:
# scale_x = 1024 / 2080
#
# scale_y tells us how much the y-coordinates changed.
# Example:
# If original height is 2336 and new height is 1024:
# scale_y = 1024 / 2336
#
# Each row gets its own scale factor, so this works even if
# your dataset contains images with different original sizes.
scale_x = TARGET_SIZE / df["width"]
scale_y = TARGET_SIZE / df["height"]


# ============================================================
# TRANSFORM BOUNDING BOX COORDINATES
# ============================================================
# Since x_min and x_max are horizontal coordinates,
# they are multiplied by scale_x.
#
# Since y_min and y_max are vertical coordinates,
# they are multiplied by scale_y.
#
# This makes the bounding boxes match the resized 1024 x 1024 image.
df["x_min"] = df["x_min"] * scale_x
df["x_max"] = df["x_max"] * scale_x

df["y_min"] = df["y_min"] * scale_y
df["y_max"] = df["y_max"] * scale_y


# ============================================================
# ROUND AND CLIP BOUNDING BOX VALUES
# ============================================================
# Bounding box coordinates are usually stored as integers.
# .round() converts values like 340.18 into 340.
#
# .clip(0, 1024) makes sure coordinates do not go below 0
# or above 1024 after scaling.
bbox_cols = ["x_min", "y_min", "x_max", "y_max"]

for col in bbox_cols:
    df[col] = df[col].round().clip(0, TARGET_SIZE).astype(int)


# ============================================================
# UPDATE IMAGE WIDTH AND HEIGHT COLUMNS
# ============================================================
# Since the new images are now 1024 x 1024,
# we update the width and height columns to reflect that.
df["width"] = TARGET_SIZE
df["height"] = TARGET_SIZE


# ============================================================
# SAVE THE TRANSFORMED DATASET
# ============================================================
# This creates a new CSV file with the transformed bounding boxes.
# Your original CSV file will not be modified.
df.to_csv(OUTPUT_CSV, index=False)


# ============================================================
# DISPLAY RESULT
# ============================================================
# Prints confirmation and previews the first few rows.
print(f"Saved transformed annotations to: {OUTPUT_CSV}")

df.head()

Saved transformed annotations to: ../data/processed/02_vinbig_transformed_bounding_boxes_1024.csv


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max,width,height,target_class_id
0,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,340,603,814,803,1024,1024,1
1,1c32170b4af4ce1a3030eb8167753b06,Pleural thickening,11,R9,253,119,382,144,1024,1024,2
2,47ed17dcb2cbeec15182ed335a8b5a9e,Nodule/Mass,8,R9,222,718,269,759,1024,1024,5
3,afb6230703512afc370f236e8fe98806,Pulmonary fibrosis,13,R9,749,536,857,679,1024,1024,3
4,18a61a07e6f5f13ebfee57fa36cd8b6f,Pulmonary fibrosis,13,R9,175,101,365,229,1024,1024,3


In [8]:
# =========================
# Clear variables to adjust
# =========================

CSV_PATH = Path("../data/processed/02_vinbig_transformed_bounding_boxes_1024.csv")

DEFAULT_IMAGE_SIZE = (1024, 1024)  # (height, width)
CLUSTER_IOU_THR = 0.60
NMS_IOU_THR = 0.50
AREA_RATIO_THRESH = 2.0
FORCE_FUSE = False
PER_ANNOTATOR = False

classes = {
    "Cardiomegaly": 1,
    "Pleural thickening": 2,
    "Pulmonary fibrosis": 3,
    "Pleural effusion": 4,
    "Nodule/Mass": 5,
    "Infiltration": 6,
    "Consolidation": 7,
    "Atelectasis": 8,
    "Pneumothorax": 9,
}

id_to_class = {v: k for k, v in classes.items()}


# =========================
# Box utility functions
# =========================

def compute_iou(boxA, boxB):
    """Compute IoU for boxes in [x1, y1, x2, y2] absolute pixel format."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0.0, xB - xA)
    inter_h = max(0.0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0.0, boxA[2] - boxA[0]) * max(0.0, boxA[3] - boxA[1])
    areaB = max(0.0, boxB[2] - boxB[0]) * max(0.0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0.0


def box_center(box):
    """Return center point (cx, cy)."""
    return ((box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0)


def box_area(box):
    """Return box area."""
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def center_distance(boxA, boxB):
    """Euclidean distance between two box centers."""
    ax, ay = box_center(boxA)
    bx, by = box_center(boxB)
    return math.hypot(ax - bx, ay - by)


def max_pairwise_center_distance(cluster_indices, boxes):
    """Largest center distance inside a cluster."""
    if len(cluster_indices) <= 1:
        return 0.0

    max_dist = 0.0
    for i, idx_a in enumerate(cluster_indices):
        for idx_b in cluster_indices[i + 1:]:
            max_dist = max(max_dist, center_distance(boxes[idx_a], boxes[idx_b]))
    return max_dist


def max_area_ratio(cluster_indices, boxes):
    """Largest/smallest area ratio inside a cluster."""
    areas = [box_area(boxes[idx]) for idx in cluster_indices if box_area(boxes[idx]) > 0]
    if len(areas) <= 1:
        return 1.0
    return max(areas) / min(areas)


# =========================
# Clustering and fusion
# =========================

def make_iou_clusters(indices, boxes, cluster_iou_thr):
    """Cluster boxes if they are same-class and overlap enough by IoU."""
    remaining = set(indices)
    clusters = []

    while remaining:
        seed = remaining.pop()
        cluster = {seed}
        queue = [seed]

        while queue:
            current = queue.pop()
            neighbors = [
                idx for idx in list(remaining)
                if compute_iou(boxes[current], boxes[idx]) >= cluster_iou_thr
            ]
            for idx in neighbors:
                remaining.remove(idx)
                cluster.add(idx)
                queue.append(idx)

        clusters.append(sorted(cluster))

    return clusters


def split_borderline_cluster(cluster_indices, boxes, scores, centroid_spread_thresh, area_ratio_thresh):
    """
    Split a cluster if it is too spatially spread out or has very different box sizes.
    This protects true separate findings from being fused just because annotators overlap loosely.
    """
    if len(cluster_indices) <= 1:
        return [cluster_indices], False

    spread = max_pairwise_center_distance(cluster_indices, boxes)
    ratio = max_area_ratio(cluster_indices, boxes)
    is_borderline = spread > centroid_spread_thresh or ratio > area_ratio_thresh

    if not is_borderline:
        return [cluster_indices], False

    sorted_indices = sorted(cluster_indices, key=lambda idx: scores[idx], reverse=True)
    subclusters = []

    for idx in sorted_indices:
        placed = False
        for subcluster in subclusters:
            center_ok = all(
                center_distance(boxes[idx], boxes[member]) <= centroid_spread_thresh
                for member in subcluster
            )
            area_ok = all(
                max(box_area(boxes[idx]), box_area(boxes[member])) /
                max(min(box_area(boxes[idx]), box_area(boxes[member])), 1e-9)
                <= area_ratio_thresh
                for member in subcluster
            )

            if center_ok and area_ok:
                subcluster.append(idx)
                placed = True
                break

        if not placed:
            subclusters.append([idx])

    return subclusters, True


def weighted_boxes_fusion(cluster_indices, boxes, scores, labels, annotator_ids, annotator_weights=None):
    """Fuse one tight cluster into a single weighted box."""
    annotator_weights = annotator_weights or {}

    weights = []
    for idx in cluster_indices:
        annotator = annotator_ids[idx]
        annotator_weight = annotator_weights.get(annotator, 1.0)
        weights.append(scores[idx] * annotator_weight)

    weights = np.array(weights, dtype=float)
    if weights.sum() <= 0:
        weights = np.ones(len(cluster_indices), dtype=float)

    cluster_boxes = np.array([boxes[idx] for idx in cluster_indices], dtype=float)
    fused_box = np.average(cluster_boxes, axis=0, weights=weights).tolist()

    fused_score = float(np.average([scores[idx] for idx in cluster_indices], weights=weights))
    label = int(labels[cluster_indices[0]])

    return {
        "box": [round(float(v), 3) for v in fused_box],
        "score": round(fused_score, 5),
        "label": label,
        "label_name": id_to_class.get(label, str(label)),
        "contributors": list(cluster_indices),
        "annotators": sorted({annotator_ids[idx] for idx in cluster_indices}, key=str),
        "review_flag": False,
    }


def nms_same_class(fused_boxes, nms_iou_thr=0.5):
    """Apply NMS only within the same class after fusion."""
    by_label = defaultdict(list)
    for item in fused_boxes:
        by_label[item["label"]].append(item)

    kept = []

    for label, items in by_label.items():
        items = sorted(items, key=lambda x: x["score"], reverse=True)

        while items:
            best = items.pop(0)
            kept.append(best)

            survivors = []
            for candidate in items:
                same_instance = compute_iou(best["box"], candidate["box"]) > nms_iou_thr

                # If either box needs review, keep it rather than silently suppressing it.
                if same_instance and not best["review_flag"] and not candidate["review_flag"]:
                    continue

                survivors.append(candidate)

            items = survivors

    return kept


# =========================
# Main fusion functions
# =========================

def fuse_boxes_for_image(
    boxes,
    scores,
    labels,
    annotator_ids,
    image_size,
    cluster_iou_thr=0.6,
    centroid_spread_thresh=None,
    area_ratio_thresh=2.0,
    nms_iou_thr=0.5,
    annotator_weights=None,
    per_annotator=False,
    force_fuse=False,
    thoracic_box=None,
):
    """
    Fuse duplicate boxes for one image while preserving separate same-class instances.

    Special rule:
    - Cardiomegaly always returns at most one output box.
    - CTR is None unless thoracic_box is provided.
    """
    if centroid_spread_thresh is None:
        height, width = image_size
        centroid_spread_thresh = 0.05 * math.hypot(height, width)

    original_count = len(boxes)

    if per_annotator:
        fused_per_annotator = []
        for annotator in sorted(set(annotator_ids), key=str):
            idxs = [i for i, a in enumerate(annotator_ids) if a == annotator]
            remap_boxes = [boxes[i] for i in idxs]
            remap_scores = [scores[i] for i in idxs]
            remap_labels = [labels[i] for i in idxs]
            remap_annotators = [annotator_ids[i] for i in idxs]

            result = fuse_boxes_for_image(
                remap_boxes,
                remap_scores,
                remap_labels,
                remap_annotators,
                image_size=image_size,
                cluster_iou_thr=cluster_iou_thr,
                centroid_spread_thresh=centroid_spread_thresh,
                area_ratio_thresh=area_ratio_thresh,
                nms_iou_thr=nms_iou_thr,
                annotator_weights=annotator_weights,
                per_annotator=False,
                force_fuse=force_fuse,
                thoracic_box=thoracic_box,
            )

            for fused in result["fused_boxes"]:
                fused["contributors"] = [idxs[i] for i in fused["contributors"]]
                fused_per_annotator.append(fused)

        return reconcile_across_annotators(
            fused_per_annotator,
            image_size=image_size,
            cluster_iou_thr=cluster_iou_thr,
            centroid_spread_thresh=centroid_spread_thresh,
            area_ratio_thresh=area_ratio_thresh,
            nms_iou_thr=nms_iou_thr,
            force_fuse=force_fuse,
            original_count=original_count,
            thoracic_box=thoracic_box,
        )

    fused_boxes = []
    merged_clusters_flagged = 0

    label_to_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        label_to_indices[int(label)].append(idx)

    for label, indices in label_to_indices.items():
        if label == classes["Cardiomegaly"]:
            continue

        iou_clusters = make_iou_clusters(indices, boxes, cluster_iou_thr)

        for cluster in iou_clusters:
            subclusters, was_borderline = split_borderline_cluster(
                cluster,
                boxes,
                scores,
                centroid_spread_thresh,
                area_ratio_thresh,
            )

            if was_borderline:
                merged_clusters_flagged += 1

            if was_borderline and force_fuse:
                fused = weighted_boxes_fusion(cluster, boxes, scores, labels, annotator_ids, annotator_weights)
                fused["review_flag"] = True
                fused_boxes.append(fused)
            else:
                for subcluster in subclusters:
                    fused = weighted_boxes_fusion(subcluster, boxes, scores, labels, annotator_ids, annotator_weights)
                    fused["review_flag"] = was_borderline
                    fused_boxes.append(fused)

    cardiomegaly_result = handle_cardiomegaly(
        boxes,
        scores,
        labels,
        annotator_ids,
        annotator_weights=annotator_weights,
        thoracic_box=thoracic_box,
    )

    if cardiomegaly_result["fused_box"] is not None:
        fused_boxes.append(cardiomegaly_result["fused_box"])
        if cardiomegaly_result["fused_box"]["review_flag"]:
            merged_clusters_flagged += 1

    fused_boxes = nms_same_class(fused_boxes, nms_iou_thr=nms_iou_thr)

    return {
        "fused_boxes": fused_boxes,
        "stats": {
            "original_count": original_count,
            "fused_count": len(fused_boxes),
            "merged_clusters_flagged": merged_clusters_flagged,
        },
        "ctr": cardiomegaly_result["ctr"],
    }


def handle_cardiomegaly(boxes, scores, labels, annotator_ids, annotator_weights=None, thoracic_box=None):
    """
    Cardiomegaly is treated as one canonical image-level finding.
    Multiple annotators can contribute, but only one final box is produced.
    """
    cardiomegaly_label = classes["Cardiomegaly"]
    indices = [i for i, label in enumerate(labels) if int(label) == cardiomegaly_label]

    if not indices:
        return {"fused_box": None, "ctr": None}

    fused = weighted_boxes_fusion(indices, boxes, scores, labels, annotator_ids, annotator_weights)
    fused["review_flag"] = len(set(annotator_ids[i] for i in indices)) > 1 and len(indices) > 1

    cardiac_width = max(0.0, fused["box"][2] - fused["box"][0])
    if thoracic_box is None:
        ctr = None
    else:
        thoracic_width = max(0.0, thoracic_box[2] - thoracic_box[0])
        ctr = cardiac_width / thoracic_width if thoracic_width > 0 else None

    return {"fused_box": fused, "ctr": ctr}


def reconcile_across_annotators(
    fused_per_annotator,
    image_size,
    cluster_iou_thr=0.6,
    centroid_spread_thresh=None,
    area_ratio_thresh=2.0,
    nms_iou_thr=0.5,
    force_fuse=False,
    original_count=None,
    thoracic_box=None,
):
    """Merge already-fused per-annotator results using the same safeguards."""
    boxes = [item["box"] for item in fused_per_annotator]
    scores = [item["score"] for item in fused_per_annotator]
    labels = [item["label"] for item in fused_per_annotator]
    annotator_ids = [
        item["annotators"][0] if item["annotators"] else "unknown"
        for item in fused_per_annotator
    ]

    result = fuse_boxes_for_image(
        boxes,
        scores,
        labels,
        annotator_ids,
        image_size=image_size,
        cluster_iou_thr=cluster_iou_thr,
        centroid_spread_thresh=centroid_spread_thresh,
        area_ratio_thresh=area_ratio_thresh,
        nms_iou_thr=nms_iou_thr,
        per_annotator=False,
        force_fuse=force_fuse,
        thoracic_box=thoracic_box,
    )

    for fused in result["fused_boxes"]:
        original_contributors = []
        original_annotators = set()

        for fused_idx in fused["contributors"]:
            original_contributors.extend(fused_per_annotator[fused_idx]["contributors"])
            original_annotators.update(fused_per_annotator[fused_idx]["annotators"])
            fused["review_flag"] = fused["review_flag"] or fused_per_annotator[fused_idx]["review_flag"]

        fused["contributors"] = sorted(set(original_contributors))
        fused["annotators"] = sorted(original_annotators, key=str)

    if original_count is not None:
        result["stats"]["original_count"] = original_count

    return result


# =========================
# Optional CSV helper
# =========================

def load_annotations_for_image(csv_path, target_image_id=None):
    """
    Loads annotations from your transformed CSV.

    Expected columns:
    - image_id
    - x_min, y_min, x_max, y_max
    - class_name or target_class_id/class_id
    - rad_id or annotator_id
    """
    rows_by_image = defaultdict(lambda: {"boxes": [], "scores": [], "labels": [], "annotator_ids": []})

    with csv_path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            image_id = (row.get("image_id") or "").strip()
            if target_image_id is not None and image_id != target_image_id:
                continue

            class_name = (row.get("class_name") or "").strip()
            label = row.get("target_class_id") or row.get("class_id") or classes.get(class_name)

            if not image_id or not label:
                continue

            try:
                box = [
                    float(row["x_min"]),
                    float(row["y_min"]),
                    float(row["x_max"]),
                    float(row["y_max"]),
                ]
            except (KeyError, TypeError, ValueError):
                continue

            rows_by_image[image_id]["boxes"].append(box)
            rows_by_image[image_id]["scores"].append(float(row.get("score") or 1.0))
            rows_by_image[image_id]["labels"].append(int(label))
            rows_by_image[image_id]["annotator_ids"].append(row.get("rad_id") or row.get("annotator_id") or "unknown")

    return rows_by_image


# =========================
# Synthetic example + tests
# =========================

image_size = (1024, 1024)

boxes = [
    [100, 100, 180, 180],  # Atelectasis instance 1, annotator A
    [130, 105, 210, 185],  # Atelectasis instance 2, nearby but should remain separate
    [500, 500, 580, 580],  # Nodule/Mass duplicate, annotator A
    [505, 505, 585, 585],  # Nodule/Mass duplicate, annotator B
    [310, 420, 700, 650],  # Cardiomegaly, annotator A
    [320, 430, 715, 660],  # Cardiomegaly, annotator B, triggers review rule
]

scores = [0.95, 0.94, 0.90, 0.85, 0.88, 0.82]

labels = [
    classes["Atelectasis"],
    classes["Atelectasis"],
    classes["Nodule/Mass"],
    classes["Nodule/Mass"],
    classes["Cardiomegaly"],
    classes["Cardiomegaly"],
]

annotator_ids = ["R1", "R2", "R1", "R2", "R1", "R2"]

result = fuse_boxes_for_image(
    boxes=boxes,
    scores=scores,
    labels=labels,
    annotator_ids=annotator_ids,
    image_size=image_size,
    cluster_iou_thr=CLUSTER_IOU_THR,
    nms_iou_thr=NMS_IOU_THR,
    area_ratio_thresh=AREA_RATIO_THRESH,
    per_annotator=PER_ANNOTATOR,
    force_fuse=FORCE_FUSE,
)

print("Fused boxes:")
for item in result["fused_boxes"]:
    print(item)

print("\nStats:")
print(result["stats"])
print("CTR:", result["ctr"])

label_counts = defaultdict(int)
for item in result["fused_boxes"]:
    label_counts[item["label_name"]] += 1

assert label_counts["Atelectasis"] == 2, "Expected two separate Atelectasis instances"
assert label_counts["Nodule/Mass"] == 1, "Expected duplicate Nodule/Mass boxes to fuse"
assert label_counts["Cardiomegaly"] == 1, "Expected at most one Cardiomegaly box"
assert any(
    item["label_name"] == "Cardiomegaly" and item["review_flag"]
    for item in result["fused_boxes"]
), "Expected Cardiomegaly to trigger review_flag"

print("\nAll synthetic assertions passed.")


# =========================
# Optional: run on one real image from CSV
# =========================
# annotations_by_image = load_annotations_for_image(CSV_PATH)
# target_image_id = next(iter(annotations_by_image))
# real = annotations_by_image[target_image_id]
#
# real_result = fuse_boxes_for_image(
#     boxes=real["boxes"],
#     scores=real["scores"],
#     labels=real["labels"],
#     annotator_ids=real["annotator_ids"],
#     image_size=DEFAULT_IMAGE_SIZE,
# )
#
# print("\nReal image:", target_image_id)
# print(real_result)

Fused boxes:
{'box': [100.0, 100.0, 180.0, 180.0], 'score': 0.95, 'label': 8, 'label_name': 'Atelectasis', 'contributors': [0], 'annotators': ['R1'], 'review_flag': False}
{'box': [130.0, 105.0, 210.0, 185.0], 'score': 0.94, 'label': 8, 'label_name': 'Atelectasis', 'contributors': [1], 'annotators': ['R2'], 'review_flag': False}
{'box': [502.429, 502.429, 582.429, 582.429], 'score': 0.87571, 'label': 5, 'label_name': 'Nodule/Mass', 'contributors': [2, 3], 'annotators': ['R1', 'R2'], 'review_flag': False}
{'box': [314.824, 424.824, 707.235, 654.824], 'score': 0.85106, 'label': 1, 'label_name': 'Cardiomegaly', 'contributors': [4, 5], 'annotators': ['R1', 'R2'], 'review_flag': True}

Stats:
{'original_count': 6, 'fused_count': 4, 'merged_clusters_flagged': 1}
CTR: None

All synthetic assertions passed.


In [9]:
INPUT_CSV = Path("../data/processed/02_vinbig_transformed_bounding_boxes_1024.csv")
OUTPUT_CSV = Path("../data/processed/03_vinbig_wbf_bounding_boxes_1024.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

annotations_by_image = load_annotations_for_image(INPUT_CSV)

with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as outfile:
    fieldnames = [
        "image_id",
        "label",
        "label_name",
        "score",
        "x_min",
        "y_min",
        "x_max",
        "y_max",
        "contributors",
        "annotators",
        "review_flag",
        "ctr",
    ]

    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()

    total_original = 0
    total_fused = 0
    total_flagged = 0

    for image_id, data in annotations_by_image.items():
        result = fuse_boxes_for_image(
            boxes=data["boxes"],
            scores=data["scores"],
            labels=data["labels"],
            annotator_ids=data["annotator_ids"],
            image_size=DEFAULT_IMAGE_SIZE,
            cluster_iou_thr=CLUSTER_IOU_THR,
            nms_iou_thr=NMS_IOU_THR,
            area_ratio_thresh=AREA_RATIO_THRESH,
            per_annotator=PER_ANNOTATOR,
            force_fuse=FORCE_FUSE,
        )

        total_original += result["stats"]["original_count"]
        total_fused += result["stats"]["fused_count"]
        total_flagged += result["stats"]["merged_clusters_flagged"]

        for item in result["fused_boxes"]:
            x_min, y_min, x_max, y_max = item["box"]

            writer.writerow({
                "image_id": image_id,
                "label": item["label"],
                "label_name": item["label_name"],
                "score": item["score"],
                "x_min": x_min,
                "y_min": y_min,
                "x_max": x_max,
                "y_max": y_max,
                "contributors": "|".join(map(str, item["contributors"])),
                "annotators": "|".join(map(str, item["annotators"])),
                "review_flag": item["review_flag"],
                "ctr": result["ctr"],
            })

print(f"Input annotations: {total_original:,}")
print(f"WBF annotations: {total_fused:,}")
print(f"Flagged clusters: {total_flagged:,}")
print(f"Output CSV: {OUTPUT_CSV.resolve()}")

Input annotations: 22,288
WBF annotations: 15,116
Flagged clusters: 1,852
Output CSV: D:\Home\Documents\GitHub\cxraide-data-pipeline\notebooks\data\processed\03_vinbig_wbf_bounding_boxes_1024.csv


In [12]:
bbox_csv_path = Path('../data/processed/03_vinbig_wbf_bounding_boxes_1024.csv')

with bbox_csv_path.open(newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    abnormality_counts = Counter(row['label_name'] for row in reader)

print(f'Total annotated abnormalities: {sum(abnormality_counts.values()):,}')
print('\nAbnormalities per classification:')
for finding_label, count in abnormality_counts.most_common():
    print(f'{finding_label}: {count:,}')


Total annotated abnormalities: 15,116

Abnormalities per classification:
Pleural thickening: 4,056
Pulmonary fibrosis: 3,364
Cardiomegaly: 2,300
Nodule/Mass: 1,865
Pleural effusion: 1,774
Infiltration: 960
Consolidation: 434
Atelectasis: 231
Pneumothorax: 132
